In [2]:
from typing import Annotated, TypedDict
from langchain_ollama import ChatOllama
from langchain_core.messages import AnyMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages 
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import  interrupt, Command
from dotenv import load_dotenv


In [3]:
load_dotenv()

True

In [4]:
llm = ChatOllama(model="llama3.2")

In [5]:
class ChatState(TypedDict): 
    messages: Annotated[list[BaseMessage], add_messages]
    

In [6]:
def chat_node(state: ChatState): 
    decision = interrupt({
        "type": "approval", 
        "reason": "Model is about to answer a user question.", 
        "question": state['messages'][-1].content,
        "instruction": "Approve this question? Yes/No",
    })

    if decision['approved'] == 'no':
        return {"messages": [AIMessage(content="Not approved.")]}
    else: 
        response = llm.invoke(state['messages'])
        return {'messages': [response]}